In [10]:
import json
import re
import os

def clean_output_tags(text):
    """
    终极版标签修复：暴力拆解后重新组装，免疫所有嵌套、重复、乱序问题
    """
    if not text:
        return text

    # 1. 釜底抽薪：无差别剔除所有现存的 <think> 和 </think> 标签，彻底打碎混乱的嵌套
    text = re.sub(r'</?think>', '', text, flags=re.IGNORECASE)
    
    # 2. 剔除所有毫无意义的空 <final></final> 标签
    text = re.sub(r'<final>\s*</final>', '', text, flags=re.IGNORECASE)
    
    # 去除首尾多余的空白和换行
    text = text.strip()
    
    # 3. 重新规范化组装
    if re.search(r'<final>', text, flags=re.IGNORECASE):
        # 如果存在真正的 <final>，在第一个 <final> 前面插入 </think>，并在全文最开头插入 <think>
        text = re.sub(r'<final>', '</think>\n<final>', text, count=1, flags=re.IGNORECASE)
        text = f"<think>\n{text}"
    else:
        # 如果压根没有 <final> 标签，说明全文都是思考过程，直接首尾包裹
        text = f"<think>\n{text}\n</think>"
        
    return text

def process_jsonl_inplace(file_path):
    """
    读取 JSONL 文件，修复标签后【原地】覆盖原文件
    """
    temp_path = file_path + ".tmp"
    fixed_count = 0
    total_count = 0

    print(f"🚀 开始终极清理并覆盖文件: {file_path}")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as infile, \
             open(temp_path, 'w', encoding='utf-8') as outfile:
             
            for line in infile:
                if not line.strip():
                    continue
                    
                total_count += 1
                data = json.loads(line)
                original_output = data.get("output", "")
                
                # 执行终极清洗
                fixed_output = clean_output_tags(original_output)
                
                if original_output != fixed_output:
                    fixed_count += 1
                    data["output"] = fixed_output
                    
                outfile.write(json.dumps(data, ensure_ascii=False) + '\n')
                
        # 全部完成后，安全替换原文件
        os.replace(temp_path, file_path)
        
        print("-" * 35)
        print("✅ 处理完成！")
        print(f"📄 总数据量: {total_count} 条")
        print(f"🔧 成功修复: {fixed_count} 条")
        print(f"💾 原文件已被成功更新！")

    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        print(f"❌ 处理过程中出现错误: {e}")

if __name__ == "__main__":
    # 直接填你的文件路径
    TARGET_FILE = '/mnt/data/zwl/verl/data/mixed_40.jsonl'  
    
    process_jsonl_inplace(TARGET_FILE)

🚀 开始终极清理并覆盖文件: /mnt/data/zwl/verl/data/mixed_40.jsonl
-----------------------------------
✅ 处理完成！
📄 总数据量: 40 条
🔧 成功修复: 40 条
💾 原文件已被成功更新！
